# ApexPINN: FastF1 data probe (PLAN T6)

Thin runner around `scripts/probe_fastf1.py`. It loads one race each from 2023, 2024, and 2025 with laps, telemetry, and weather, then writes:

- `experiments/data_probe/probe_results.json` (machine-readable, four-field records)
- `reports/data_probe.md` (human-readable report)

Works on **Google Colab** (outputs go to Google Drive) and **Kaggle** (outputs go to `/kaggle/working`). No GPU needed: pick a CPU runtime.

Kaggle differences:
- Turn on **Settings > Internet** (needs a phone-verified account), or FastF1 cannot download.
- There is no Drive. Results land in `/kaggle/working/apexpinn/`; download them from the Output panel or after **Save Version**.
- If the Kaggle image is not Python 3.12, `pip install -e .` is refused by `requires-python`. The notebook then installs only `fastf1` and `pyyaml` and runs from `src/` via `PYTHONPATH`, which is all the probe needs.
- For a private repo, store a GitHub token as a Kaggle secret (Add-ons > Secrets) or Colab secret (key icon) named `GH_TOKEN`.

In [ ]:
# Parameters
REPO_URL = ""  # e.g. "https://github.com/<user>/apexpinn.git"; leave empty to use REPO_ZIP
REPO_BRANCH = "main"
REPO_ZIP = "apexpinn_src.zip"  # fallback: zip of pyproject.toml, README.md, src/, scripts/, configs/
INSTALL_MODE = "full"  # "full" = pip install -e . (needs Python 3.12); "probe" = fastf1 + pyyaml only
# SEASON:GRAND_PRIX:CIRCUIT_ID, circuit ids as in configs/decisions_v1.yaml
RACES = ["2023:Spain:barcelona", "2024:Bahrain:bahrain", "2025:Japan:suzuka"]
SAVE_CACHE_TO_PERSISTENT = False  # copy the FastF1 cache to Drive / working dir afterwards (hundreds of MB)

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_ROOT = Path("/content/drive/MyDrive/apexpinn")
    REPO_DIR = Path("/content/apexpinn")
    CACHE_DIR = Path("/content/.fastf1_cache")  # local disk: SQLite caches are slow and fragile on Drive
    ZIP_SEARCH = [OUT_ROOT / REPO_ZIP]
elif IN_KAGGLE:
    OUT_ROOT = Path("/kaggle/working/apexpinn")
    REPO_DIR = Path("/tmp/apexpinn")  # outside /kaggle/working so the repo is not saved as output
    CACHE_DIR = Path("/tmp/.fastf1_cache")
    ZIP_SEARCH = sorted(Path("/kaggle/input").glob(f"**/{REPO_ZIP}"))  # upload the zip as a Dataset
else:
    OUT_ROOT = Path.cwd() / "probe_outputs"
    REPO_DIR = Path.cwd() / "apexpinn"
    CACHE_DIR = Path.cwd() / ".fastf1_cache"
    ZIP_SEARCH = [Path.cwd() / REPO_ZIP]

OUT_ROOT.mkdir(parents=True, exist_ok=True)
runtime = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "other"
print("runtime:", runtime, "| python", sys.version.split()[0])
print("outputs ->", OUT_ROOT)

In [ ]:
import shutil
import subprocess
import zipfile


def github_token():
    try:
        if IN_COLAB:
            from google.colab import userdata

            return userdata.get("GH_TOKEN")
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient

            return UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        return None
    return os.environ.get("GH_TOKEN")


if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if REPO_URL:
    url = REPO_URL
    token = github_token()
    if token and url.startswith("https://github.com/"):
        url = url.replace("https://", f"https://x-access-token:{token}@", 1)
    # The URL may hold a token, so it is not printed.
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, url, str(REPO_DIR)], check=True)
else:
    zips = [p for p in ZIP_SEARCH if p.exists()]
    if not zips:
        raise FileNotFoundError(f"Set REPO_URL or upload {REPO_ZIP}; searched {ZIP_SEARCH}")
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(REPO_DIR)
    # Accept a zip whose content sits in one top-level folder.
    inner = [p for p in REPO_DIR.iterdir() if p.is_dir()]
    if not (REPO_DIR / "pyproject.toml").exists() and len(inner) == 1:
        REPO_DIR = inner[0]
assert (REPO_DIR / "scripts" / "probe_fastf1.py").exists(), "repo is missing scripts/probe_fastf1.py"
print("repo at", REPO_DIR)

In [ ]:
env = dict(os.environ)
mode = INSTALL_MODE if sys.version_info[:2] == (3, 12) else "probe"
if mode == "full":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
else:
    print(f"Python {sys.version.split()[0]} or INSTALL_MODE='probe': installing fastf1 + pyyaml only")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastf1>=3.4", "pyyaml>=6.0"], check=True)
    env["PYTHONPATH"] = str(REPO_DIR / "src") + os.pathsep + env.get("PYTHONPATH", "")
check = "import apexpinn.data.probe, fastf1; print('fastf1', fastf1.__version__)"
subprocess.run([sys.executable, "-c", check], check=True, env=env)

In [ ]:
json_out = OUT_ROOT / "experiments" / "data_probe" / "probe_results.json"
report_out = OUT_ROOT / "reports" / "data_probe.md"
cmd = [sys.executable, str(REPO_DIR / "scripts" / "probe_fastf1.py"), "--cache-dir", str(CACHE_DIR),
       "--json-out", str(json_out), "--report-out", str(report_out)]
for race in RACES:
    cmd += ["--race", race]
proc = subprocess.run(cmd, env=env, cwd=REPO_DIR)
print("exit code", proc.returncode, "(1 means every race failed; see the log above)")

if SAVE_CACHE_TO_PERSISTENT and CACHE_DIR.exists():
    shutil.copytree(CACHE_DIR, OUT_ROOT / ".fastf1_cache", dirs_exist_ok=True)

In [ ]:
from IPython.display import Markdown, display

display(Markdown(report_out.read_text(encoding="utf-8")))
print("JSON:", json_out)
print("Report:", report_out)

## Bring the results back

Copy `probe_results.json` and `data_probe.md` into the local repo at `experiments/data_probe/` and `reports/`. O1-O8 are finalized from them in `configs/decisions_v1.yaml`, and O5 is reviewed before any model results are seen.

To build `apexpinn_src.zip` on Windows from the repo root (no git remote needed):

```powershell
Compress-Archive -Force -Path pyproject.toml, README.md, src, scripts, configs -DestinationPath apexpinn_src.zip
```

Upload it to `MyDrive/apexpinn/` (Colab) or as a private Kaggle Dataset (Kaggle).